# Production Simulation on Kaggle

Simulasi production untuk proyek **highway-lane-occupancy** di Kaggle, menggantikan
Colab. Notebook ini memproses semua klip dalam satu Kaggle Dataset, menghasilkan
artefak batch (video, events, tracks, reports, evidence), lalu opsional me-replay
events/tracks ke webhook pipeline.

**Alur:** cek lingkungan -> clone/install -> siapkan dataset + secret -> batch
pipeline -> replay -> unduh bundle.

> Kaggle: Settings -> Accelerator -> **GPU T4 x2** (opsional), Internet **ON**.
> Default pipeline adalah **single pass**. Jika hanya butuh anotasi satu video,
> gunakan `notebooks/colab_annotate_tmp_kaggle.ipynb`.

In [ ]:
import os
import shutil
from pathlib import Path

import torch

print("torch      :", torch.__version__)
print("CUDA ready :", torch.cuda.is_available())
print("GPU count  :", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}    : {props.name} ({props.total_memory / 1e9:.1f} GB)")

for folder in ("/kaggle/input", "/kaggle/working"):
    if Path(folder).exists():
        usage = shutil.disk_usage(folder)
        print(f"{folder}: {usage.free / 1e9:.1f} GB free")

## 1. Clone dan install

Kaggle sudah membawa PyTorch CUDA; jangan install wheel torch CPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-occupancy"

if os.path.basename(os.getcwd()) != PROJECT:
    if not Path(PROJECT).is_dir():
        subprocess.run(["git", "clone", REPO, PROJECT], check=True)
    os.chdir(PROJECT)

ROOT = Path.cwd()
SRC = ROOT / "src"
assert (SRC / "lane_analytics").is_dir(), f"Repo tidak lengkap: {SRC}"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"],
    check=True,
)

import ultralytics
from lane_analytics.colab import activate

print("repo         :", ROOT)
print("ultralytics  :", ultralytics.__version__)
print("activate     :", activate.__module__)

## 2. Siapkan input dan secret

Upload video sebagai **Kaggle Dataset** (mis. `highway-videos`), lalu sesuaikan
`DATASET_DIR`. Notebook akan mencari file `.mp4` di dalamnya dan membuat satu ZIP agar
`activate()` bisa memproses semuanya.

Untuk replay ke pipeline:

1. Tambahkan secret `WEBHOOK_URL` (wajib) dan `WEBHOOK_TOKEN` (opsional) di
   **Add-ons -> Secrets**.
2. Set `REPLAY = True` dan `DRY_RUN = False`.
3. Pastikan URL webhook bisa diakses publik dari Kaggle (ngrok/Cloudflare Tunnel/VM).

Untuk uji aman tanpa jaringan, biarkan `REPLAY = False` atau `DRY_RUN = True`.

In [ ]:
from pathlib import Path
import zipfile

# --- ubah bila perlu -------------------------------------------------------
DATASET_DIR = "/kaggle/input/highway-videos"  # folder dataset Kaggle
VIDEO_ZIP = ""            # opsional: path zip yang sudah ada di dataset
BASE_TIMESTAMP = ""       # "" = infer dari nama file / chain durasi
MAX_VIDEOS = 0            # 0 = semua
FRAME_STRIDE = 1
IMGSZ = 960
RENDER_VIDEO = True
REPLAY = False            # True = kirim ke webhook
DRY_RUN = True            # True = tulis batch ke output/replay, tanpa HTTP
SPEED = 5.0               # 1 = real time, 5 = 5x
WINDOW_SECONDS = 5.0
# ---------------------------------------------------------------------------

dataset = Path(DATASET_DIR)
videos = sorted(
    path for path in dataset.rglob("*") if path.suffix.lower() in {".mp4", ".mkv", ".mov", ".avi", ".m4v", ".ts"}
) if dataset.exists() else []

print("Dataset :", dataset, "exists:", dataset.exists())
print("Videos  :", len(videos))
for path in videos[:20]:
    print(f"  - {path} ({path.stat().st_size / 1e6:.1f} MB)")
if len(videos) > 20:
    print(f"  ... {len(videos) - 20} more")

if not videos and not VIDEO_ZIP:
    print()
    print("Isi /kaggle/input:")
    for path in sorted(Path("/kaggle/input").glob("*")):
        print(" -", path)
    raise FileNotFoundError("Tidak ada video dan VIDEO_ZIP kosong.")

if VIDEO_ZIP:
    archive = Path(VIDEO_ZIP)
else:
    archive = Path("/kaggle/working/input_videos.zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_STORED) as zf:
        for path in videos:
            zf.write(path, path.name)

print("Archive :", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")

## 3. Jalankan batch pipeline + replay

`activate()` menjalankan pipeline single-pass untuk setiap klip, menyusun
`base_timestamp` (dari nama file atau rantai durasi), lalu me-replay events/tracks ke
webhook bila diminta. Output batch menjadi *source of truth*.

In [ ]:
from lane_analytics.colab import activate

device = "0" if torch.cuda.is_available() else "cpu"

print("Device        :", device)
print("Replay        :", REPLAY)
print("Dry run       :", DRY_RUN)
print("Videos        :", len(videos))
print()

result = activate(
    source="drive_zip",
    zip_path=archive,
    video_dir="data/raw",
    max_videos=MAX_VIDEOS or None,
    base_timestamp=BASE_TIMESTAMP or None,
    device=device,
    imgsz=IMGSZ,
    frame_stride=FRAME_STRIDE,
    write_video=RENDER_VIDEO,
    replay=REPLAY,
    speed=SPEED,
    window_seconds=WINDOW_SECONDS,
    dry_run=DRY_RUN,
)

print()
print(result)
for item in result.videos:
    print(
        f"- {item['video']} | status={item['status']} | "
        f"events={item.get('events', '-')} | vehicles={item.get('vehicles', '-')} | "
        f"hogging={item.get('hogging', '-')}"
    )
    if item.get("error"):
        print("  error:", item["error"])

## 4. Package hasil

Bundle berisi `manifest.json` + seluruh isi `output/` (video anotasi, events, tracks,
reports, evidence, dan batch replay dry-run bila aktif). Unduh dari tab **Output**
Kaggle sebelum sesi berakhir.

In [ ]:
import json
import zipfile
from pathlib import Path

out = Path("output")
work_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
bundle = work_dir / "production_bundle.zip"
manifest_path = work_dir / "production_manifest.json"

manifest = result.as_dict()
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, "manifest.json")
    for path in sorted(out.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(out.parent))

print(f"Manifest : {manifest_path}")
print(f"Bundle   : {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
print()
with zipfile.ZipFile(bundle) as zf:
    for name in zf.namelist()[:30]:
        print(" -", name)
    if len(zf.namelist()) > 30:
        print(f" ... {len(zf.namelist()) - 30} more")

## 5. Catatan production

- **Single pass adalah default.** Notebook ini memakai `activate()` -> `process_video`,
  bukan mode parallel. Untuk 2xT4, gunakan notebook anotasi dengan `WORKERS = 2` dan
  `DEVICES = "0,1"`; untuk batch produksi ini single pass lebih sederhana dan sudah
  cukup cepat.
- **Kaggle bukan scheduler.** Tidak ada cron native. Untuk simulasi production
  berulang, picu notebook lewat Kaggle API/CI, atau pindahkan job ke VM/container
  dengan cron di produksi nyata.
- **Batas sesi.** Kaggle membatasi durasi sesi dan kuota GPU; commit notebook
  (**Save & Run All**) agar output tersimpan setelah sesi berhenti.
- **Secret.** Jangan hardcode token. Gunakan `WEBHOOK_URL`/`WEBHOOK_TOKEN` di Kaggle
  Secrets; `activate()` membacanya otomatis.
- **Idempotensi.** Replay bersifat at-least-once. Receiver/pipeline wajib melakukan
  dedupe berdasarkan `event_id`/`run_id`.
- **Base timestamp.** Untuk produksi, isi `BASE_TIMESTAMP` atau nama file yang memuat
  waktu overlay. Timestamp yang salah menggeser seluruh event.
- **Kontrak data.** Events/tracks tetap mengikuti `docs/schema/lane_event.schema.json`
  dan `docs/HANDOFF_DATA_PIPELINE_DASHBOARD.md`.